## Player Clustering

The goal of this section is cluster players from the CSA T20 Challenge based on their batting and bowling attributes. With these clusterings, we can look for players of skillsets and even find players who closely match a requirement that the team needs.

## Converting to CSV

In [1]:
import json
import pandas as pd
from collections.abc import Mapping, Sequence

with open("ctc_json/1444765.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# start from the innings list (keeps original nesting)
df = pd.DataFrame(data["innings"])

def explode_and_normalize_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Repeatedly explode the first column that contains lists (non-string),
    then normalize any dicts produced into columns. Continue until no list
    cells remain. Finally expand any remaining dict columns.
    """
    def is_list_cell(x):
        return isinstance(x, Sequence) and not isinstance(x, (str, bytes, bytearray, dict))
    # explode list columns until none remain
    while True:
        list_cols = [c for c in df.columns if df[c].apply(is_list_cell).any()]
        if not list_cols:
            break
        col = list_cols[0]
        df = df.explode(col).reset_index(drop=True)
        # if exploded values are dicts, expand them into prefixed columns
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            norm = pd.json_normalize(df[col].fillna({}).tolist())
            if not norm.empty:
                norm = norm.add_prefix(f"{col}_")
                df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
            else:
                df = df.drop(columns=[col])
    # expand any remaining dict columns
    dict_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, dict)).any()]
    for col in dict_cols:
        norm = pd.json_normalize(df[col].fillna({}).tolist())
        if not norm.empty:
            norm = norm.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
        else:
            df = df.drop(columns=[col])
    return df

df_expanded = explode_and_normalize_all(df)
print("rows:", df_expanded.shape[0], "cols:", df_expanded.shape[1])
df_expanded.head()

rows: 260 cols: 18


,team,overs_over,powerplays_from,powerplays_to,powerplays_type,overs_deliveries_batter,overs_deliveries_bowler,overs_deliveries_non_striker,overs_deliveries_runs.batter,overs_deliveries_runs.extras,overs_deliveries_runs.total,overs_deliveries_extras.noballs,overs_deliveries_extras.wides,overs_deliveries_extras.byes,overs_deliveries_extras.legbyes,overs_deliveries_wickets_player_out,overs_deliveries_wickets_kind,overs_deliveries_wickets_fielders_name
0,Titans,0,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Titans,0,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Titans,0,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Titans,0,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Titans,0,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,1,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df_expanded = explode_and_normalize_all(df)

# detect the over column (common names produced by the explode/normalize logic)
over_col = next((c for c in df_expanded.columns if c in ("overs_over", "over") or c.endswith("_over")), None)
# detect an innings identifier (team / inning column)
team_candidates = ["team", "innings_team", "team_name", "inning_team"]
team_col = next((c for c in team_candidates if c in df_expanded.columns), None)

# fallback: try to match a column containing one of the teams from the original JSON
if team_col is None:
    teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
    for c in df_expanded.columns:
        if df_expanded[c].dropna().isin(teams).any():
            team_col = c
            break

# compute ball_number (1-based): restart per innings if team_col found, otherwise per over or global
if over_col is None:
    df_expanded["ball_number"] = range(1, len(df_expanded) + 1)
else:
    if team_col:
        df_expanded["ball_number"] = df_expanded.groupby([team_col, over_col]).cumcount() + 1
    else:
        df_expanded["ball_number"] = df_expanded.groupby(over_col).cumcount() + 1

# move ball_number to immediate right of the over column (if both exist)
if over_col in df_expanded.columns and "ball_number" in df_expanded.columns:
    cols = list(df_expanded.columns)
    try:
        over_idx = cols.index(over_col)
        # remove ball_number then insert right after over_col
        cols.remove("ball_number")
        cols.insert(over_idx + 1, "ball_number")
        df_expanded = df_expanded[cols]
    except ValueError:
        # if something unexpected happens, leave column order unchanged
        pass

print("using over_col =", over_col, "team_col =", team_col)
print("rows:", df_expanded.shape[0], "cols:", df_expanded.shape[1])
df_expanded.head()

using over_col = overs_over team_col = team
rows: 260 cols: 19


,team,overs_over,ball_number,powerplays_from,powerplays_to,powerplays_type,overs_deliveries_batter,overs_deliveries_bowler,overs_deliveries_non_striker,overs_deliveries_runs.batter,overs_deliveries_runs.extras,overs_deliveries_runs.total,overs_deliveries_extras.noballs,overs_deliveries_extras.wides,overs_deliveries_extras.byes,overs_deliveries_extras.legbyes,overs_deliveries_wickets_player_out,overs_deliveries_wickets_kind,overs_deliveries_wickets_fielders_name
0,Titans,0,1,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Titans,0,2,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Titans,0,3,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Titans,0,4,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Titans,0,5,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,1,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_expanded = df_expanded.rename(columns={"overs_over": "over"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_batter": "striker"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_bowler": "bowler"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_non_striker": "non_striker"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.batter": "batter_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.extras": "extra_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.total": "total_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.wides": "wides"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.legbyes": "legbyes"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.noballs": "noballs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.by": "review_by"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.umpire": "review_umpire"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.batter": "review_batter"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.decision": "review_decision"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.type": "review_type"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.umpires_call": "umpires_call"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_player_out": "player_out"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_kind": "wicket_kind"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_fielders_name": "fielders_name"})

df_expanded.head(10)


,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,extra_runs,total_runs,noballs,wides,overs_deliveries_extras.byes,legbyes,player_out,wicket_kind,fielders_name
0,Titans,0,1,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Titans,0,2,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Titans,0,3,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Titans,0,4,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Titans,0,5,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,1,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN
5,Titans,0,6,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Titans,0,7,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,6,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Titans,1,1,0.1,5.9,mandatory,R Moonsamy,KT Maphaka,LG Pretorius,4,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Titans,1,2,0.1,5.9,mandatory,R Moonsamy,KT Maphaka,LG Pretorius,0,0,0,NaN,NaN,NaN,NaN,R Moonsamy,bowled,NaN
9,Titans,1,3,0.1,5.9,mandatory,KD Petersen,KT Maphaka,LG Pretorius,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
team_candidates = ["team", "innings_team", "team_name", "inning_team"]
team_col = next((c for c in team_candidates if c in df_expanded.columns), None)
if team_col is None:
    teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
    for c in df_expanded.columns:
        if df_expanded[c].dropna().isin(teams).any():
            team_col = c
            break

# ensure total_runs exists and is numeric
if "total_runs" not in df_expanded.columns:
    raise KeyError("total_runs column not found in df_expanded")

df_expanded["total_runs"] = pd.to_numeric(df_expanded["total_runs"], errors="coerce").fillna(0).astype(int)

# cumulative sum per innings (resets when innings/team changes)
if team_col:
    df_expanded["current_team_total"] = df_expanded.groupby(team_col)["total_runs"].cumsum()
else:
    # fallback: global cumulative sum if no innings identifier found
    df_expanded["current_team_total"] = df_expanded["total_runs"].cumsum()

df_expanded.head(10)

,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,extra_runs,total_runs,noballs,wides,overs_deliveries_extras.byes,legbyes,player_out,wicket_kind,fielders_name,current_team_total
0,Titans,0,1,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Titans,0,2,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Titans,0,3,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Titans,0,4,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Titans,0,5,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,1,5,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5
5,Titans,0,6,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
6,Titans,0,7,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,6,0,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
7,Titans,1,1,0.1,5.9,mandatory,R Moonsamy,KT Maphaka,LG Pretorius,4,0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19
8,Titans,1,2,0.1,5.9,mandatory,R Moonsamy,KT Maphaka,LG Pretorius,0,0,0,NaN,NaN,NaN,NaN,R Moonsamy,bowled,NaN,19
9,Titans,1,3,0.1,5.9,mandatory,KD Petersen,KT Maphaka,LG Pretorius,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19


In [5]:
df_expanded.to_csv("expanded_match_data.csv", index=False)

In [6]:
from pathlib import Path
import json
import pandas as pd
from collections.abc import Mapping, Sequence

# list the files you want to convert (edit as needed)
files = [
    Path("ctc_json/1444765.json"),
    Path("ctc_json/1444766.json"),
    Path("ctc_json/1444768.json"),
    Path("ctc_json/1444770.json"),
    Path("ctc_json/1444772.json"),
    Path("ctc_json/1444773.json"),
    Path("ctc_json/1444774.json"),
    Path("ctc_json/1444776.json"),
    Path("ctc_json/1444777.json"),
    Path("ctc_json/1444778.json"),
    Path("ctc_json/1444779.json"),
    Path("ctc_json/1444780.json"),
    Path("ctc_json/1444781.json"),
    Path("ctc_json/1444783.json"),
    Path("ctc_json/1444784.json"),
    Path("ctc_json/1444785.json"),
    Path("ctc_json/1444786.json"),
    Path("ctc_json/1444787.json"),
    Path("ctc_json/1444789.json"),
    Path("ctc_json/1444790.json"),
    Path("ctc_json/1444791.json"),
    Path("ctc_json/1444792.json"),
    Path("ctc_json/1444793.json"),
    Path("ctc_json/1444794.json"),
    Path("ctc_json/1444795.json"),
    Path("ctc_json/1444796.json"),
]

out_dir = Path("ctc_csv_output")
out_dir.mkdir(parents=True, exist_ok=True)

# optional rename map (adjust if you changed earlier mappings)
rename_map = {
    "overs_over": "over",
    "overs_deliveries_batter": "striker",
    "overs_deliveries_bowler": "bowler",
    "overs_deliveries_non_striker": "non_striker",
    "overs_deliveries_runs.batter": "batter_runs",
    "overs_deliveries_runs.extras": "extra_runs",
    "overs_deliveries_runs.total": "total_runs",
    "overs_deliveries_extras.wides": "wides",
    "overs_deliveries_extras.legbyes": "legbyes",
    "overs_deliveries_extras.noballs": "noballs",
    "overs_deliveries_review.by": "review_by",
    "overs_deliveries_review.umpire": "review_umpire",
    "overs_deliveries_review.batter": "review_batter",
    "overs_deliveries_review.decision": "review_decision",
    "overs_deliveries_review.type": "review_type",
    "overs_deliveries_review.umpires_call": "umpires_call",
    "overs_deliveries_wickets_player_out": "player_out",
    "overs_deliveries_wickets_kind": "wicket_kind",
    "overs_deliveries_wickets_fielders_name": "fielders_name",
}

def explode_and_normalize_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Repeatedly explode the first column that contains lists (non-string),
    then normalize any dicts produced into columns. Continue until no list
    cells remain. Finally expand any remaining dict columns.
    """
    def is_list_cell(x):
        return isinstance(x, Sequence) and not isinstance(x, (str, bytes, bytearray, dict))
    # explode list columns until none remain
    while True:
        list_cols = [c for c in df.columns if df[c].apply(is_list_cell).any()]
        if not list_cols:
            break
        col = list_cols[0]
        df = df.explode(col).reset_index(drop=True)
        # if exploded values are dicts, expand them into prefixed columns
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            norm = pd.json_normalize(df[col].fillna({}).tolist())
            if not norm.empty:
                norm = norm.add_prefix(f"{col}_")
                df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
            else:
                df = df.drop(columns=[col])
    # expand any remaining dict columns
    dict_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, dict)).any()]
    for col in dict_cols:
        norm = pd.json_normalize(df[col].fillna({}).tolist())
        if not norm.empty:
            norm = norm.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
        else:
            df = df.drop(columns=[col])
    return df

for p in files:
    if not p.exists():
        print("skipping (not found):", p)
        continue
    try:
        with p.open("r", encoding="utf-8") as f:
            data = json.load(f)

        # base dataframe from innings
        df = pd.DataFrame(data.get("innings", []))

        # explode/normalize using the helper defined earlier in the notebook
        df_expanded = explode_and_normalize_all(df)

        # apply safe renames
        to_rename = {k: v for k, v in rename_map.items() if k in df_expanded.columns}
        if to_rename:
            df_expanded = df_expanded.rename(columns=to_rename)

        # detect team & over columns (same logic as earlier cells)
        team_candidates = ["team", "innings_team", "team_name", "inning_team"]
        team_col = next((c for c in team_candidates if c in df_expanded.columns), None)
        if team_col is None:
            teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
            for c in df_expanded.columns:
                if df_expanded[c].dropna().isin(teams).any():
                    team_col = c
                    break

        over_col = next((c for c in df_expanded.columns if c in ("overs_over", "over") or c.endswith("_over")), None)

        # compute 1-based ball_number (restart per innings) and position it after over_col
        if over_col is None:
            df_expanded["ball_number"] = range(1, len(df_expanded) + 1)
        else:
            if team_col:
                df_expanded["ball_number"] = df_expanded.groupby([team_col, over_col]).cumcount() + 1
            else:
                df_expanded["ball_number"] = df_expanded.groupby(over_col).cumcount() + 1
            if over_col in df_expanded.columns:
                cols = list(df_expanded.columns)
                try:
                    over_idx = cols.index(over_col)
                    cols.remove("ball_number")
                    cols.insert(over_idx + 1, "ball_number")
                    df_expanded = df_expanded[cols]
                except ValueError:
                    pass

        # compute current_team_total from total_runs (reset per innings)
        if "total_runs" in df_expanded.columns:
            df_expanded["total_runs"] = pd.to_numeric(df_expanded["total_runs"], errors="coerce").fillna(0).astype(int)
            if team_col:
                df_expanded["current_team_total"] = df_expanded.groupby(team_col)["total_runs"].cumsum()
            else:
                df_expanded["current_team_total"] = df_expanded["total_runs"].cumsum()
        else:
            df_expanded["current_team_total"] = 0

        # write one CSV per JSON input
        out_path = out_dir / (p.stem + ".csv")
        df_expanded.to_csv(out_path, index=False)
        print("saved:", out_path)
    except Exception as e:
        print("failed:", p, "->", e)
# ...existing code...

saved: ctc_csv_output/1444765.csv
saved: ctc_csv_output/1444766.csv
saved: ctc_csv_output/1444768.csv
saved: ctc_csv_output/1444770.csv
saved: ctc_csv_output/1444772.csv
saved: ctc_csv_output/1444773.csv
saved: ctc_csv_output/1444774.csv
saved: ctc_csv_output/1444776.csv
saved: ctc_csv_output/1444777.csv
saved: ctc_csv_output/1444778.csv
saved: ctc_csv_output/1444779.csv
saved: ctc_csv_output/1444780.csv
saved: ctc_csv_output/1444781.csv
saved: ctc_csv_output/1444783.csv
saved: ctc_csv_output/1444784.csv
saved: ctc_csv_output/1444785.csv
saved: ctc_csv_output/1444786.csv
saved: ctc_csv_output/1444787.csv
saved: ctc_csv_output/1444789.csv
saved: ctc_csv_output/1444790.csv
saved: ctc_csv_output/1444791.csv
saved: ctc_csv_output/1444792.csv
saved: ctc_csv_output/1444793.csv
saved: ctc_csv_output/1444794.csv
saved: ctc_csv_output/1444795.csv
saved: ctc_csv_output/1444796.csv


## Player Stats

In [ ]:
from pathlib import Path
import pandas as pd

csv_dir = Path("ctc_csv_output")
csv_files = sorted(csv_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {csv_dir.resolve()}")

dfs = []
for p in csv_files:
    try:
        df = pd.read_csv(p)
        df["source_file"] = p.name  # optional: keep track of origin
        dfs.append(df)
    except Exception as e:
        print(f"warning: failed to read {p.name} -> {e}")

# concatenate into a single DataFrame
all_ctc_matches_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# remove second of back-to-back identical run-out readings (within same source_file)
all_ctc_matches_df["player_out"] = all_ctc_matches_df["player_out"].fillna("").astype(str)
wicket_norm = all_ctc_matches_df["wicket_kind"].fillna("").astype(str).str.strip().str.lower()

same_source = all_ctc_matches_df["source_file"] == all_ctc_matches_df["source_file"].shift(1)
same_player_out = all_ctc_matches_df["player_out"] == all_ctc_matches_df["player_out"].shift(1)
prev_runout = wicket_norm.shift(1) == "run out"
curr_runout = wicket_norm == "run out"

consecutive_duplicate_runout = same_source & same_player_out & prev_runout & curr_runout

if consecutive_duplicate_runout.any():
    all_ctc_matches_df = all_ctc_matches_df[~consecutive_duplicate_runout].reset_index(drop=True)

# all_ctc_matches_df["venue"] = all_ctc_matches_df["source_file"].map(venue_map).astype("object")
# all_ctc_matches_df["avg_first_innings_score"] = pd.to_numeric(all_ctc_matches_df["venue"].map(avg_first_innings_map), errors="coerce")

# quick sanity checks
print("files read:", len(dfs), "rows total:", len(all_ctc_matches_df))
all_ctc_matches_df.head()


files read: 26 rows total: 6179


,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,...,current_team_total,source_file,overs_deliveries_wickets_fielders_substitute,overs_deliveries_over,overs_deliveries_replacements.role_in,overs_deliveries_replacements.role_out,overs_deliveries_replacements.role_reason,overs_deliveries_replacements.role_role,absent_hurt,overs_deliveries_runs.non_boundary
0,Titans,0,1,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,...,0,1444765.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Titans,0,2,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,...,0,1444765.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Titans,0,3,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,...,0,1444765.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Titans,0,4,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,0,...,0,1444765.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Titans,0,5,0.1,5.9,mandatory,LG Pretorius,Codi Yusuf,R Moonsamy,4,...,5,1444765.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
all_ctc_matches_df.to_csv("all_ctc_matches.csv", index=False)

In [12]:
all_ctc_matches_df = pd.read_csv("all_ctc_matches.csv")

cols = ["striker", "bowler", "non_striker"]

# Distinct values per column
distinct_by_column = {
    c: sorted(
        all_ctc_matches_df[c]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s.ne("")]
        .unique()
        .tolist()
    )
    for c in cols
}

for c in cols:
    print(f"{c}: {len(distinct_by_column[c])} distinct values")

# If you also want one combined distinct player list across all 3 columns
all_distinct_players = sorted(
    pd.unique(
        all_ctc_matches_df[cols]
        .apply(lambda s: s.dropna().astype(str).str.strip())
        .replace("", pd.NA)
        .stack(dropna=True)
    ).tolist()
)

print(f"combined distinct players: {len(all_distinct_players)}")

print(all_distinct_players)

striker: 123 distinct values
bowler: 84 distinct values
non_striker: 117 distinct values
combined distinct players: 137
['A Cloete', 'A Gqamane', 'A Mgijima', 'A Mnyaka', 'A Mokgakane', 'A Nortje', 'A Simelane', 'AL Phehlukwayo', 'AM Phangiso', 'AR Swanepoel', 'Abdullah Bayoumy', 'B Capell', 'B Parsons', 'B Swanepoel', 'B Xenxe', 'BC Fortuin', 'BE Hendricks', 'BG Porteous', 'BS Makhanya', 'C Bosch', 'C Esterhuizen', 'C Fortuin', 'C Seleka', 'CJ Dala', 'CJ King', 'Codi Yusuf', 'D Brevis', 'D Ferreira', 'D Forrester', 'D Jansen', 'D Potgieter', 'D Smith', 'DA Miller', 'DG Bedingham', 'DL Piedt', 'DM Dupavillon', 'Dayyaan Galiem', 'E Bosch', 'E Jones', 'EM Moore', 'FD Adams', 'G Coetzee', 'G Kaplan', 'G Roelofsen', 'G Tarr', 'GA Stuurman', 'GF Linde', 'GG Peters', 'GJ Snyman', 'GL Cloete', 'H Klaasen', 'H Viljoen', 'HE van der Dussen', 'I Manack', 'J Dawood', 'J Hermann', 'J James', 'J King', 'J Pillay', 'J Richards', 'J du Plessis', 'J van Dyk', 'JA Bird', 'JF Smith', 'JN Malan', 'JP Kin

/var/folders/qh/dg_1llp91pjb4yfbj6kj3cc80000gn/T/ipykernel_36057/3663779574.py:28: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(dropna=True)


In [14]:
import re

batter_dfs = {}
for name in all_distinct_players:
    df = all_ctc_matches_df[all_ctc_matches_df["striker"] == name].copy()
    batter_dfs[name] = df
    var_name = "df_" + re.sub(r"\W+", "_", name.strip()).lower()
    globals()[var_name] = df  # optional: create global var for quick access

# summary
print(f"created {len(batter_dfs)} batter dataframes")
for name, df in batter_dfs.items():
    print(name, "rows:", len(df))

created 137 batter dataframes
A Cloete rows: 0
A Gqamane rows: 26
A Mgijima rows: 52
A Mnyaka rows: 5
A Mokgakane rows: 24
A Nortje rows: 0
A Simelane rows: 20
AL Phehlukwayo rows: 52
AM Phangiso rows: 51
AR Swanepoel rows: 39
Abdullah Bayoumy rows: 54
B Capell rows: 6
B Parsons rows: 83
B Swanepoel rows: 50
B Xenxe rows: 6
BC Fortuin rows: 7
BE Hendricks rows: 16
BG Porteous rows: 30
BS Makhanya rows: 80
C Bosch rows: 11
C Esterhuizen rows: 129
C Fortuin rows: 62
C Seleka rows: 12
CJ Dala rows: 17
CJ King rows: 0
Codi Yusuf rows: 7
D Brevis rows: 19
D Ferreira rows: 134
D Forrester rows: 71
D Jansen rows: 7
D Potgieter rows: 103
D Smith rows: 91
DA Miller rows: 14
DG Bedingham rows: 4
DL Piedt rows: 7
DM Dupavillon rows: 13
Dayyaan Galiem rows: 122
E Bosch rows: 22
E Jones rows: 86
EM Moore rows: 198
FD Adams rows: 6
G Coetzee rows: 27
G Kaplan rows: 80
G Roelofsen rows: 136
G Tarr rows: 59
GA Stuurman rows: 6
GF Linde rows: 101
GG Peters rows: 2
GJ Snyman rows: 119
GL Cloete rows: 55

In [16]:
out_dir = Path("csa_t20_challenge/batters")
out_dir.mkdir(parents=True, exist_ok=True)

for name, df in batter_dfs.items():
    safe_name = re.sub(r"\W+", "_", name.strip()).lower().strip("_")
    if not safe_name:
        safe_name = "unknown_batter"
    out_path = out_dir / f"{safe_name}.csv"
    df.to_csv(out_path, index=False)
    print("saved:", out_path)

saved: csa_t20_challenge/batters/a_cloete.csv
saved: csa_t20_challenge/batters/a_gqamane.csv
saved: csa_t20_challenge/batters/a_mgijima.csv
saved: csa_t20_challenge/batters/a_mnyaka.csv
saved: csa_t20_challenge/batters/a_mokgakane.csv
saved: csa_t20_challenge/batters/a_nortje.csv
saved: csa_t20_challenge/batters/a_simelane.csv
saved: csa_t20_challenge/batters/al_phehlukwayo.csv
saved: csa_t20_challenge/batters/am_phangiso.csv
saved: csa_t20_challenge/batters/ar_swanepoel.csv
saved: csa_t20_challenge/batters/abdullah_bayoumy.csv
saved: csa_t20_challenge/batters/b_capell.csv
saved: csa_t20_challenge/batters/b_parsons.csv
saved: csa_t20_challenge/batters/b_swanepoel.csv
saved: csa_t20_challenge/batters/b_xenxe.csv
saved: csa_t20_challenge/batters/bc_fortuin.csv
saved: csa_t20_challenge/batters/be_hendricks.csv
saved: csa_t20_challenge/batters/bg_porteous.csv
saved: csa_t20_challenge/batters/bs_makhanya.csv
saved: csa_t20_challenge/batters/c_bosch.csv
saved: csa_t20_challenge/batters/c_est

In [15]:
bowler_dfs = {}
for name in all_distinct_players:
    df = all_ctc_matches_df[all_ctc_matches_df["bowler"] == name].copy()
    bowler_dfs[name] = df
    var_name = "df_" + re.sub(r"\W+", "_", name.strip()).lower()
    globals()[var_name] = df  # optional: create global var for quick access

# summary
print(f"created {len(bowler_dfs)} bowler dataframes")
for name, df in bowler_dfs.items():
    print(name, "rows:", len(df))

created 137 bowler dataframes
A Cloete rows: 62
A Gqamane rows: 67
A Mgijima rows: 0
A Mnyaka rows: 30
A Mokgakane rows: 0
A Nortje rows: 48
A Simelane rows: 113
AL Phehlukwayo rows: 60
AM Phangiso rows: 135
AR Swanepoel rows: 30
Abdullah Bayoumy rows: 0
B Capell rows: 0
B Parsons rows: 24
B Swanepoel rows: 100
B Xenxe rows: 33
BC Fortuin rows: 89
BE Hendricks rows: 159
BG Porteous rows: 0
BS Makhanya rows: 6
C Bosch rows: 124
C Esterhuizen rows: 0
C Fortuin rows: 0
C Seleka rows: 121
CJ Dala rows: 113
CJ King rows: 71
Codi Yusuf rows: 95
D Brevis rows: 6
D Ferreira rows: 56
D Forrester rows: 52
D Jansen rows: 69
D Potgieter rows: 108
D Smith rows: 0
DA Miller rows: 0
DG Bedingham rows: 0
DL Piedt rows: 50
DM Dupavillon rows: 66
Dayyaan Galiem rows: 187
E Bosch rows: 21
E Jones rows: 178
EM Moore rows: 0
FD Adams rows: 19
G Coetzee rows: 100
G Kaplan rows: 0
G Roelofsen rows: 0
G Tarr rows: 0
GA Stuurman rows: 104
GF Linde rows: 124
GG Peters rows: 27
GJ Snyman rows: 55
GL Cloete rows:

In [17]:
out_dir = Path("csa_t20_challenge/bowlers")
out_dir.mkdir(parents=True, exist_ok=True)

for name, df in bowler_dfs.items():
    safe_name = re.sub(r"\W+", "_", name.strip()).lower().strip("_")
    if not safe_name:
        safe_name = "unknown_bowler"
    out_path = out_dir / f"{safe_name}.csv"
    df.to_csv(out_path, index=False)
    print("saved:", out_path)

saved: csa_t20_challenge/bowlers/a_cloete.csv
saved: csa_t20_challenge/bowlers/a_gqamane.csv
saved: csa_t20_challenge/bowlers/a_mgijima.csv
saved: csa_t20_challenge/bowlers/a_mnyaka.csv
saved: csa_t20_challenge/bowlers/a_mokgakane.csv
saved: csa_t20_challenge/bowlers/a_nortje.csv
saved: csa_t20_challenge/bowlers/a_simelane.csv
saved: csa_t20_challenge/bowlers/al_phehlukwayo.csv
saved: csa_t20_challenge/bowlers/am_phangiso.csv
saved: csa_t20_challenge/bowlers/ar_swanepoel.csv
saved: csa_t20_challenge/bowlers/abdullah_bayoumy.csv
saved: csa_t20_challenge/bowlers/b_capell.csv
saved: csa_t20_challenge/bowlers/b_parsons.csv
saved: csa_t20_challenge/bowlers/b_swanepoel.csv
saved: csa_t20_challenge/bowlers/b_xenxe.csv
saved: csa_t20_challenge/bowlers/bc_fortuin.csv
saved: csa_t20_challenge/bowlers/be_hendricks.csv
saved: csa_t20_challenge/bowlers/bg_porteous.csv
saved: csa_t20_challenge/bowlers/bs_makhanya.csv
saved: csa_t20_challenge/bowlers/c_bosch.csv
saved: csa_t20_challenge/bowlers/c_est

In [2]:
import numpy as np
from pathlib import Path
import pandas as pd
import re

batters_dir = Path("csa_t20_challenge/batters")
files = sorted(batters_dir.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No files in {batters_dir.resolve()}")

rows = []
run_cols = ["batter_runs"]
over_candidates = ["over"]

def over_to_phase(o):
    try:
        if np.isnan(o):
            return None
        o = int(float(o))
    except Exception:
        return None
    if 0 <= o <= 5:
        return "powerplay"
    if 6 <= o <= 14:
        return "middle"
    if 15 <= o <= 19:
        return "death"
    return None

phases = ["powerplay", "middle", "death"]

for p in files:
    df = pd.read_csv(p)

    # batter name
    if "striker" in df.columns and df["striker"].dropna().any():
        batter_name = df["striker"].dropna().astype(str).iloc[0]
    else:
        batter_name = re.sub(r"[_\-]+", " ", p.stem).strip()

    # runs column
    run_col = next((c for c in run_cols if c in df.columns), None)
    if run_col is None:
        runs = pd.Series([0] * len(df), index=df.index)
    else:
        runs = pd.to_numeric(df[run_col], errors="coerce").fillna(0)

    # over values
    over_col = next((c for c in df.columns if c in over_candidates or c.endswith("_over") or c == "over"), None)
    if over_col:
        over_vals = pd.to_numeric(df[over_col], errors="coerce")
        phase_series = over_vals.apply(over_to_phase)
    else:
        phase_series = pd.Series([None] * len(df), index=df.index)

    # overall stats
    total_runs = int(runs.sum())
    entries = int(len(df))
    if "player_out" in df.columns:
        dismissals = int((df["player_out"].fillna("").astype(str) == batter_name).sum())
    else:
        dismissals = 0

    average = float(total_runs) / dismissals if dismissals > 0 else float(total_runs)
    strike_rate = float(total_runs * 100) / entries if entries > 0 else 0.0

    # one output row per batter
    out = {
        "file": p.name,
        "player": batter_name,
        "total_runs": total_runs,
        "entries": entries,
        "dismissals": dismissals,
        "average": average,
        "strike_rate": strike_rate,
    }

    # add 15 flattened phase columns
    for phase in phases:
        mask = phase_series == phase
        phase_runs = int(runs[mask].sum())
        phase_entries = int(mask.sum())

        if "player_out" in df.columns:
            phase_dismissals = int(
                (df.loc[mask, "player_out"].fillna("").astype(str) == batter_name).sum()
            )
        else:
            phase_dismissals = 0

        phase_average = float(phase_runs) / phase_dismissals if phase_dismissals > 0 else float(phase_runs)
        phase_strike_rate = float(phase_runs * 100) / phase_entries if phase_entries > 0 else 0.0

        out[f"{phase}_runs"] = phase_runs
        out[f"{phase}_entries"] = phase_entries
        out[f"{phase}_dismissals"] = phase_dismissals
        out[f"{phase}_average"] = phase_average
        out[f"{phase}_strike_rate"] = phase_strike_rate

    rows.append(out)

stats_df = pd.DataFrame(rows)

# optional: column order
stats_df = stats_df[
    [
        "file", "player", "total_runs", "entries", "dismissals", "average", "strike_rate",
        "powerplay_runs", "powerplay_entries", "powerplay_dismissals", "powerplay_average", "powerplay_strike_rate",
        "middle_runs", "middle_entries", "middle_dismissals", "middle_average", "middle_strike_rate",
        "death_runs", "death_entries", "death_dismissals", "death_average", "death_strike_rate",
    ]
]

stats_df.head()

,file,player,total_runs,entries,dismissals,average,strike_rate,powerplay_runs,powerplay_entries,powerplay_dismissals,...,middle_runs,middle_entries,middle_dismissals,middle_average,middle_strike_rate,death_runs,death_entries,death_dismissals,death_average,death_strike_rate
0,a_cloete.csv,a cloete,0,0,0,0.000000,0.000000,0,0,0,...,0,0,0,0.0,0.000000,0,0,0,0.0,0.00
1,a_gqamane.csv,A Gqamane,34,26,3,11.333333,130.769231,0,0,0,...,26,18,1,26.0,144.444444,8,8,2,4.0,100.00
2,a_mgijima.csv,A Mgijima,70,52,3,23.333333,134.615385,0,0,0,...,61,46,2,30.5,132.608696,9,6,1,9.0,150.00
3,a_mnyaka.csv,A Mnyaka,2,5,0,2.000000,40.000000,0,0,0,...,2,5,0,2.0,40.000000,0,0,0,0.0,0.00
4,a_mokgakane.csv,A Mokgakane,26,24,3,8.666667,108.333333,7,8,1,...,0,0,0,0.0,0.000000,19,16,2,9.5,118.75


In [12]:
stats_df.to_csv("csa_t20_challenge/batters_stats.csv", index=False)

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import pandas as pd

# Filter: keep only players with entries >= 15
stats_df = stats_df[stats_df["entries"] >= 15].reset_index(drop=True)
print(f"Filtered to {len(stats_df)} players with entries >= 15")

feature_cols = [
    "total_runs",
    "dismissals",
    "average",
    "strike_rate",
    "powerplay_average",
    "powerplay_strike_rate",
    "middle_average",
    "middle_strike_rate",
    "death_average",
    "death_strike_rate",
]

# 1) Build feature matrix
X = stats_df[feature_cols].copy()

# 2) Handle missing values
X = X.fillna(X.median(numeric_only=True))

# 3) Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4) Fit K-means with k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
stats_df["cluster"] = kmeans.fit_predict(X_scaled)

# 5) View results
display(stats_df[["player", "cluster"] + feature_cols].head(10))

# 6) Cluster summary
cluster_summary = stats_df.groupby("cluster")[feature_cols].mean().round(2)
display(cluster_summary)

# 7) Cluster sizes
print(stats_df["cluster"].value_counts().sort_index())

Filtered to 86 players with entries >= 15


,player,cluster,total_runs,dismissals,average,strike_rate,powerplay_average,powerplay_strike_rate,middle_average,middle_strike_rate,death_average,death_strike_rate
0,A Gqamane,1,34,3,11.333333,130.769231,0.0,0.000000,26.000000,144.444444,4.0,100.000000
1,A Mgijima,1,70,3,23.333333,134.615385,0.0,0.000000,30.500000,132.608696,9.0,150.000000
2,A Mokgakane,4,26,3,8.666667,108.333333,7.0,87.500000,0.000000,0.000000,9.5,118.750000
3,A Simelane,1,21,3,7.000000,105.000000,0.0,0.000000,5.666667,89.473684,4.0,400.000000
4,Abdullah Bayoumy,4,49,3,16.333333,90.740741,0.0,0.000000,16.000000,72.727273,11.0,103.125000
5,AL Phehlukwayo,1,69,2,34.500000,132.692308,0.0,0.000000,19.000000,76.000000,50.0,185.185185
6,AM Phangiso,1,72,2,36.000000,141.176471,0.0,0.000000,24.000000,104.347826,48.0,171.428571
7,AR Swanepoel,1,53,3,17.666667,135.897436,0.0,0.000000,29.000000,170.588235,12.0,109.090909
8,B Parsons,2,97,6,16.166667,116.867470,10.6,117.777778,37.000000,119.354839,7.0,100.000000
9,B Swanepoel,0,98,3,32.666667,196.000000,45.0,180.000000,53.000000,220.833333,0.0,0.000000


,total_runs,dismissals,average,strike_rate,powerplay_average,powerplay_strike_rate,middle_average,middle_strike_rate,death_average,death_strike_rate
cluster,,,,,,,,,,
0,158.29,4.86,32.26,148.12,54.95,147.68,30.65,144.37,0.14,4.76
1,64.63,2.89,24.84,135.34,1.34,10.13,22.61,115.98,24.55,162.90
2,82.46,4.29,22.50,111.11,15.96,97.32,25.54,116.82,5.11,50.52
3,177.00,4.36,44.59,136.02,22.24,82.17,52.80,131.43,31.19,181.89
4,22.77,1.68,14.43,86.80,2.75,33.82,7.39,53.10,8.98,72.17


cluster
0     7
1    19
2    24
3    14
4    22
Name: count, dtype: int64


In [14]:
# Save batter clusters to CSV
stats_df[["player", "cluster"] + feature_cols].to_csv("csa_t20_challenge/batter_clusters.csv", index=False)
print("Saved batter_clusters.csv")

Saved batter_clusters.csv


In [4]:
bowlers_dir = Path("csa_t20_challenge/bowlers")
files = sorted(bowlers_dir.glob("*.csv"))
if not files:
    raise FileNotFoundError(f"No files in {bowlers_dir.resolve()}")

b_rows = []
run_cols = ["batter_runs"]
over_candidates = ["over"]
phases = ["powerplay", "middle", "death"]

def over_to_phase(o):
    try:
        if np.isnan(o):
            return None
        o = int(float(o))
    except Exception:
        return None
    if 0 <= o <= 5:
        return "powerplay"
    if 6 <= o <= 14:
        return "middle"
    if 15 <= o <= 19:
        return "death"
    return None

for p in files:
    dfb = pd.read_csv(p)

    # bowler name
    if "bowler" in dfb.columns and dfb["bowler"].dropna().any():
        bowler_name = dfb["bowler"].dropna().astype(str).iloc[0]
    else:
        bowler_name = re.sub(r"[_\-]+", " ", p.stem).strip()

    # runs conceded (off bat)
    run_col = next((c for c in run_cols if c in dfb.columns), None)
    runs = pd.to_numeric(dfb[run_col], errors="coerce").fillna(0) if run_col else pd.Series([0] * len(dfb), index=dfb.index)

    # extras
    wides = pd.to_numeric(dfb.get("wides", pd.Series([np.nan] * len(dfb), index=dfb.index)), errors="coerce")
    noballs = pd.to_numeric(dfb.get("noballs", pd.Series([np.nan] * len(dfb), index=dfb.index)), errors="coerce")
    wides_num = wides.fillna(0)
    noballs_num = noballs.fillna(0)

    # legal delivery mask
    extra_mask = pd.Series(False, index=dfb.index)
    for col in ["wides", "noballs"]:
        if col in dfb.columns:
            extra_mask = extra_mask | (dfb[col].notna() & (dfb[col].astype(str).str.strip() != ""))
    legal_mask = ~extra_mask

    # per-ball conceded
    row_conceded = runs + wides_num + noballs_num

    # over -> phase
    over_col = next((c for c in dfb.columns if c in over_candidates or c.endswith("_over") or c == "over"), None)
    if over_col:
        over_vals = pd.to_numeric(dfb[over_col], errors="coerce")
        phase_series = over_vals.apply(over_to_phase)
    else:
        phase_series = pd.Series([None] * len(dfb), index=dfb.index)

    # overall stats
    total_runs_conceded = float(row_conceded.sum())
    legal_deliveries = int(legal_mask.sum())
    total_wickets = int((dfb.get("player_out").notna() & (dfb.get("player_out").astype(str).str.strip() != "")).sum())
    economy = round((6.0 * total_runs_conceded) / legal_deliveries, 2) if legal_deliveries > 0 else float("nan")
    average = round(total_runs_conceded / total_wickets, 2) if total_wickets > 0 else 0.0

    out = {
        "file": p.name,
        "player": bowler_name,
        "total_runs": total_runs_conceded,
        "deliveries": legal_deliveries,
        "wickets": total_wickets,
        "economy": economy,
        "average": average,
    }

    # flattened phase metrics (15 columns)
    for phase in phases:
        mask = phase_series == phase
        phase_runs = float(row_conceded[mask].sum())
        phase_deliveries = int(legal_mask[mask].sum())
        phase_wickets = int(
            (dfb.loc[mask, "player_out"].notna() & (dfb.loc[mask, "player_out"].astype(str).str.strip() != "")).sum()
        )
        phase_economy = round((6.0 * phase_runs) / phase_deliveries, 2) if phase_deliveries > 0 else float("nan")
        phase_average = round(phase_runs / phase_wickets, 2) if phase_wickets > 0 else 0.0

        out[f"{phase}_runs"] = phase_runs
        out[f"{phase}_deliveries"] = phase_deliveries
        out[f"{phase}_wickets"] = phase_wickets
        out[f"{phase}_economy"] = phase_economy
        out[f"{phase}_average"] = phase_average

    b_rows.append(out)

bowlers_stats_df = pd.DataFrame(b_rows)

# optional: fixed column order
bowlers_stats_df = bowlers_stats_df[
    [
        "file", "player", "total_runs", "deliveries", "wickets", "economy", "average",
        "powerplay_runs", "powerplay_deliveries", "powerplay_wickets", "powerplay_economy", "powerplay_average",
        "middle_runs", "middle_deliveries", "middle_wickets", "middle_economy", "middle_average",
        "death_runs", "death_deliveries", "death_wickets", "death_economy", "death_average",
    ]
]

bowlers_stats_df.head(50)

,file,player,total_runs,deliveries,wickets,economy,average,powerplay_runs,powerplay_deliveries,powerplay_wickets,...,middle_runs,middle_deliveries,middle_wickets,middle_economy,middle_average,death_runs,death_deliveries,death_wickets,death_economy,death_average
0,a_cloete.csv,A Cloete,90.0,60,2,9.00,45.00,0.0,0,0,...,61.0,36,0,10.17,0.00,29.0,24,2,7.25,14.50
1,a_gqamane.csv,A Gqamane,127.0,64,6,11.91,21.17,0.0,0,0,...,73.0,35,4,12.51,18.25,54.0,29,2,11.17,27.00
2,a_mgijima.csv,a mgijima,0.0,0,0,NaN,0.00,0.0,0,0,...,0.0,0,0,NaN,0.00,0.0,0,0,NaN,0.00
3,a_mnyaka.csv,A Mnyaka,38.0,30,1,7.60,38.00,0.0,0,0,...,38.0,30,1,7.60,38.00,0.0,0,0,NaN,0.00
4,a_mokgakane.csv,a mokgakane,0.0,0,0,NaN,0.00,0.0,0,0,...,0.0,0,0,NaN,0.00,0.0,0,0,NaN,0.00
5,a_nortje.csv,A Nortje,78.0,48,1,9.75,78.00,37.0,18,0,...,34.0,24,1,8.50,34.00,7.0,6,0,7.00,0.00
6,a_simelane.csv,A Simelane,172.0,108,12,9.56,14.33,33.0,18,3,...,89.0,60,6,8.90,14.83,50.0,30,3,10.00,16.67
7,abdullah_bayoumy.csv,abdullah bayoumy,0.0,0,0,NaN,0.00,0.0,0,0,...,0.0,0,0,NaN,0.00,0.0,0,0,NaN,0.00
8,al_phehlukwayo.csv,AL Phehlukwayo,67.0,60,2,6.70,33.50,0.0,0,0,...,59.0,54,2,6.56,29.50,8.0,6,0,8.00,0.00
9,am_phangiso.csv,AM Phangiso,138.0,132,7,6.27,19.71,19.0,18,0,...,114.0,108,7,6.33,16.29,5.0,6,0,5.00,0.00


In [5]:
bowlers_stats_df.to_csv("csa_t20_challenge/bowlers_stats.csv", index=False)

In [6]:
# Filter: keep only bowlers with deliveries >= 15
bowlers_stats_df_filtered = bowlers_stats_df[bowlers_stats_df["deliveries"] >= 15].reset_index(drop=True)
print(f"Filtered to {len(bowlers_stats_df_filtered)} bowlers with deliveries >= 15")

bowler_feature_cols = [
    "total_runs",
    "wickets",
    "economy",
    "average",
    "powerplay_economy",
    "powerplay_average",
    "middle_economy",
    "middle_average",
    "death_economy",
    "death_average",
]

# 1) Build feature matrix
X_bowlers = bowlers_stats_df_filtered[bowler_feature_cols].copy()

# 2) Handle missing values
X_bowlers = X_bowlers.fillna(X_bowlers.median(numeric_only=True))

# 3) Scale features
scaler_bowlers = StandardScaler()
X_bowlers_scaled = scaler_bowlers.fit_transform(X_bowlers)

# 4) Fit K-means with k=5
kmeans_bowlers = KMeans(n_clusters=5, random_state=42, n_init=20)
bowlers_stats_df_filtered["cluster"] = kmeans_bowlers.fit_predict(X_bowlers_scaled)

# 5) View results
display(bowlers_stats_df_filtered[["player", "cluster"] + bowler_feature_cols].head(10))

# 6) Cluster summary
bowler_cluster_summary = bowlers_stats_df_filtered.groupby("cluster")[bowler_feature_cols].mean().round(2)
display(bowler_cluster_summary)

# 7) Cluster sizes
print(bowlers_stats_df_filtered["cluster"].value_counts().sort_index())

Filtered to 75 bowlers with deliveries >= 15


,player,cluster,total_runs,wickets,economy,average,powerplay_economy,powerplay_average,middle_economy,middle_average,death_economy,death_average
0,A Cloete,2,90.0,2,9.00,45.00,NaN,0.0,10.17,0.00,7.25,14.50
1,A Gqamane,1,127.0,6,11.91,21.17,NaN,0.0,12.51,18.25,11.17,27.00
2,A Mnyaka,4,38.0,1,7.60,38.00,NaN,0.0,7.60,38.00,NaN,0.00
3,A Nortje,4,78.0,1,9.75,78.00,12.33,0.0,8.50,34.00,7.00,0.00
4,A Simelane,1,172.0,12,9.56,14.33,11.00,11.0,8.90,14.83,10.00,16.67
5,AL Phehlukwayo,3,67.0,2,6.70,33.50,NaN,0.0,6.56,29.50,8.00,0.00
6,AM Phangiso,3,138.0,7,6.27,19.71,6.33,0.0,6.33,16.29,5.00,0.00
7,AR Swanepoel,2,40.0,0,8.00,0.00,NaN,0.0,8.50,0.00,6.00,0.00
8,B Parsons,3,25.0,1,6.25,25.00,NaN,0.0,6.33,19.00,6.00,0.00
9,B Swanepoel,1,159.0,5,9.94,31.80,8.90,44.5,8.00,8.00,12.40,31.00


,total_runs,wickets,economy,average,powerplay_economy,powerplay_average,middle_economy,middle_average,death_economy,death_average
cluster,,,,,,,,,,
0,137.00,1.00,8.56,137.00,9.00,90.00,9.00,0.00,7.60,0.00
1,159.60,7.40,9.05,23.39,7.98,25.27,8.96,13.85,10.89,26.14
2,49.40,1.40,8.39,25.47,5.23,4.90,10.06,6.17,7.62,4.17
3,85.65,5.04,6.50,17.96,6.95,4.92,6.33,15.84,6.55,5.37
4,101.00,2.75,8.37,46.38,9.15,9.80,8.30,44.27,8.63,4.22


cluster
0     1
1    20
2    15
3    23
4    16
Name: count, dtype: int64


In [ ]:
bowlers_stats_df_filtered[["player", "cluster"] + bowler_feature_cols].to_csv("csa_t20_challenge/bowler_clusters.csv", index=False)
print("Saved bowler_clusters.csv")

In [5]:
# Function to find closest player in batter clusters
def find_closest_batter(query_stats):
    """
    Find the closest batter to a given set of stats.
    
    query_stats: dict with keys like 'total_runs', 'dismissals', 'average', 'strike_rate', etc.
    Example: {'total_runs': 200, 'average': 25, 'strike_rate': 120, 'dismissals': 8,
              'powerplay_average': 30, 'powerplay_strike_rate': 130,
              'middle_average': 25, 'middle_strike_rate': 115,
              'death_average': 35, 'death_strike_rate': 145}
    """
    # Convert query to same feature order
    query_values = [query_stats.get(col, X[col].median()) for col in feature_cols]
    query_scaled = scaler.transform([query_values])
    
    # Calculate distances to all players
    distances = ((X_scaled - query_scaled[0]) ** 2).sum(axis=1) ** 0.5
    closest_idx = distances.argmin()
    
    result = stats_df.iloc[closest_idx].to_dict()
    result['distance'] = float(distances[closest_idx])
    return result


# Function to find closest player in bowler clusters
def find_closest_bowler(query_stats):
    """
    Find the closest bowler to a given set of stats.
    
    query_stats: dict with keys like 'total_runs', 'economy', 'average', 'wickets', etc.
    Example: {'total_runs': 150, 'economy': 8.5, 'average': 30, 'wickets': 5,
              'powerplay_economy': 9, 'powerplay_average': 28,
              'middle_economy': 8, 'middle_average': 32,
              'death_economy': 9.5, 'death_average': 25}
    """
    # Convert query to same feature order
    query_values = [query_stats.get(col, X_bowlers[col].median()) for col in bowler_feature_cols]
    query_scaled = scaler_bowlers.transform([query_values])
    
    # Calculate distances to all bowlers
    distances = ((X_bowlers_scaled - query_scaled[0]) ** 2).sum(axis=1) ** 0.5
    closest_idx = distances.argmin()
    
    result = bowlers_stats_df.iloc[closest_idx].to_dict()
    result['distance'] = float(distances[closest_idx])
    return result

In [7]:
from ipywidgets import FloatSlider, IntSlider, Button, Output, VBox, HBox, Label
import ipywidgets as widgets

# Create sliders for batter search (adjust ranges based on your data)
print("=== BATTER SEARCH ===")
batter_inputs = {
    'total_runs': IntSlider(min=0, max=500, step=10, value=200, description='Total Runs:'),
    'dismissals': IntSlider(min=0, max=50, step=1, value=8, description='Dismissals:'),
    'average': FloatSlider(min=0, max=100, step=1, value=25, description='Average:'),
    'strike_rate': FloatSlider(min=0, max=200, step=5, value=120, description='Strike Rate:'),
    'powerplay_average': FloatSlider(min=0, max=80, step=1, value=30, description='PP Avg:'),
    'powerplay_strike_rate': FloatSlider(min=0, max=200, step=5, value=130, description='PP SR:'),
    'middle_average': FloatSlider(min=0, max=80, step=1, value=25, description='Mid Avg:'),
    'middle_strike_rate': FloatSlider(min=0, max=200, step=5, value=115, description='Mid SR:'),
    'death_average': FloatSlider(min=0, max=80, step=1, value=35, description='Death Avg:'),
    'death_strike_rate': FloatSlider(min=0, max=200, step=5, value=145, description='Death SR:'),
}

batter_output = Output()
batter_search_btn = Button(description='Find Closest Batter', button_style='info')

def on_batter_search_click(b):
    with batter_output:
        batter_output.clear_output()
        query = {k: v.value for k, v in batter_inputs.items()}
        result = find_closest_batter(query)
        print(f"Closest Batter: {result['player']}")
        print(f"Distance: {result['distance']:.4f} (lower = closer match)")
        print("\nStats:")
        for col in feature_cols:
            print(f"  {col}: {result.get(col, 'N/A')}")

batter_search_btn.on_click(on_batter_search_click)

# Arrange in a grid
batter_grid = VBox([
    HBox([batter_inputs['total_runs'], batter_inputs['dismissals']]),
    HBox([batter_inputs['average'], batter_inputs['strike_rate']]),
    HBox([batter_inputs['powerplay_average'], batter_inputs['powerplay_strike_rate']]),
    HBox([batter_inputs['middle_average'], batter_inputs['middle_strike_rate']]),
    HBox([batter_inputs['death_average'], batter_inputs['death_strike_rate']]),
    batter_search_btn,
    batter_output
])

display(batter_grid)

# Create sliders for bowler search
print("\n=== BOWLER SEARCH ===")
bowler_inputs = {
    'total_runs': IntSlider(min=0, max=500, step=10, value=150, description='Total Runs:'),
    'wickets': IntSlider(min=0, max=50, step=1, value=5, description='Wickets:'),
    'economy': FloatSlider(min=0, max=15, step=0.25, value=8.5, description='Economy:'),
    'average': FloatSlider(min=0, max=100, step=1, value=30, description='Average:'),
    'powerplay_economy': FloatSlider(min=0, max=15, step=0.25, value=9, description='PP Econ:'),
    'powerplay_average': FloatSlider(min=0, max=100, step=1, value=28, description='PP Avg:'),
    'middle_economy': FloatSlider(min=0, max=15, step=0.25, value=8, description='Mid Econ:'),
    'middle_average': FloatSlider(min=0, max=100, step=1, value=32, description='Mid Avg:'),
    'death_economy': FloatSlider(min=0, max=15, step=0.25, value=9.5, description='Death Econ:'),
    'death_average': FloatSlider(min=0, max=100, step=1, value=25, description='Death Avg:'),
}

bowler_output = Output()
bowler_search_btn = Button(description='Find Closest Bowler', button_style='success')

def on_bowler_search_click(b):
    with bowler_output:
        bowler_output.clear_output()
        query = {k: v.value for k, v in bowler_inputs.items()}
        result = find_closest_bowler(query)
        print(f"Closest Bowler: {result['player']}")
        print(f"Distance: {result['distance']:.4f} (lower = closer match)")
        print("\nStats:")
        for col in bowler_feature_cols:
            print(f"  {col}: {result.get(col, 'N/A')}")

bowler_search_btn.on_click(on_bowler_search_click)

# Arrange in a grid
bowler_grid = VBox([
    HBox([bowler_inputs['total_runs'], bowler_inputs['wickets']]),
    HBox([bowler_inputs['economy'], bowler_inputs['average']]),
    HBox([bowler_inputs['powerplay_economy'], bowler_inputs['powerplay_average']]),
    HBox([bowler_inputs['middle_economy'], bowler_inputs['middle_average']]),
    HBox([bowler_inputs['death_economy'], bowler_inputs['death_average']]),
    bowler_search_btn,
    bowler_output
])

display(bowler_grid)

=== BATTER SEARCH ===



=== BOWLER SEARCH ===


In [8]:
from ipywidgets import FloatSlider, IntSlider, Button, Output, VBox, HBox, Label, Tab
import ipywidgets as widgets

print("=== BATTER SEARCH BY PHASE ===")

# Updated functions to return all players with distance < 3
def find_close_batters(query_stats, max_distance=2.5):
    """
    Find all batters within max_distance of the given stats.
    """
    query_values = [query_stats.get(col, X[col].median()) for col in feature_cols]
    query_scaled = scaler.transform([query_values])
    
    distances = ((X_scaled - query_scaled[0]) ** 2).sum(axis=1) ** 0.5
    close_mask = distances < max_distance
    
    results = []
    for idx in np.where(close_mask)[0]:
        result = stats_df.iloc[idx].to_dict()
        result['distance'] = float(distances[idx])
        results.append(result)
    
    # Sort by distance
    results.sort(key=lambda x: x['distance'])
    return results


def find_close_bowlers(query_stats, max_distance=2.5):
    """
    Find all bowlers within max_distance of the given stats.
    """
    query_values = [query_stats.get(col, X_bowlers[col].median()) for col in bowler_feature_cols]
    query_scaled = scaler_bowlers.transform([query_values])
    
    distances = ((X_bowlers_scaled - query_scaled[0]) ** 2).sum(axis=1) ** 0.5
    close_mask = distances < max_distance
    
    results = []
    for idx in np.where(close_mask)[0]:
        result = bowlers_stats_df_filtered.iloc[idx].to_dict()
        result['distance'] = float(distances[idx])
        results.append(result)
    
    # Sort by distance
    results.sort(key=lambda x: x['distance'])
    return results


# Tab 1: Overall Stats (Runs, Avg, SR)
overall_inputs = {
    'total_runs': IntSlider(min=0, max=500, step=10, value=200, description='Total Runs:'),
    'average': FloatSlider(min=0, max=100, step=1, value=25, description='Average:'),
    'strike_rate': FloatSlider(min=0, max=200, step=5, value=120, description='Strike Rate:'),
}

overall_output = Output()
overall_btn = Button(description='Find Batters (distance < 3)', button_style='info')

def on_overall_search(b):
    with overall_output:
        overall_output.clear_output()
        query = {k: v.value for k, v in overall_inputs.items()}
        # Fill in other features with median
        for col in feature_cols:
            if col not in query:
                query[col] = X[col].median()
        results = find_close_batters(query)
        
        if not results:
            print("No batters found within distance 2.5")
        else:
            print(f"Found {len(results)} batter(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Total Runs: {result.get('total_runs', 'N/A')}")
                print(f"   Average: {result.get('average', 'N/A'):.2f}")
                print(f"   Strike Rate: {result.get('strike_rate', 'N/A'):.2f}")

overall_btn.on_click(on_overall_search)

overall_tab = VBox([
    HBox([overall_inputs['total_runs'], overall_inputs['average']]),
    HBox([overall_inputs['strike_rate']]),
    overall_btn,
    overall_output
])

# Tab 2: Powerplay Phase
pp_inputs = {
    'powerplay_average': FloatSlider(min=0, max=80, step=1, value=30, description='PP Average:'),
    'powerplay_strike_rate': FloatSlider(min=0, max=200, step=5, value=130, description='PP Strike Rate:'),
}

pp_output = Output()
pp_btn = Button(description='Find Batters (distance < 2.5)', button_style='info')

def on_pp_search(b):
    with pp_output:
        pp_output.clear_output()
        query = {k: v.value for k, v in pp_inputs.items()}
        for col in feature_cols:
            if col not in query:
                query[col] = X[col].median()
        results = find_close_batters(query)
        
        if not results:
            print("No batters found within distance 2.5")
        else:
            print(f"Found {len(results)} batter(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   PP Runs: {result.get('powerplay_runs', 'N/A')}")
                print(f"   PP Average: {result.get('powerplay_average', 'N/A'):.2f}")
                print(f"   PP Strike Rate: {result.get('powerplay_strike_rate', 'N/A'):.2f}")

pp_btn.on_click(on_pp_search)

pp_tab = VBox([
    HBox([pp_inputs['powerplay_average'], pp_inputs['powerplay_strike_rate']]),
    pp_btn,
    pp_output
])

# Tab 3: Middle Phase
mid_inputs = {
    'middle_average': FloatSlider(min=0, max=80, step=1, value=25, description='Middle Average:'),
    'middle_strike_rate': FloatSlider(min=0, max=200, step=5, value=115, description='Middle Strike Rate:'),
}

mid_output = Output()
mid_btn = Button(description='Find Batters (distance < 3)', button_style='info')

def on_mid_search(b):
    with mid_output:
        mid_output.clear_output()
        query = {k: v.value for k, v in mid_inputs.items()}
        for col in feature_cols:
            if col not in query:
                query[col] = X[col].median()
        results = find_close_batters(query)
        
        if not results:
            print("No batters found within distance 2.5")
        else:
            print(f"Found {len(results)} batter(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Middle Runs: {result.get('middle_runs', 'N/A')}")
                print(f"   Middle Average: {result.get('middle_average', 'N/A'):.2f}")
                print(f"   Middle Strike Rate: {result.get('middle_strike_rate', 'N/A'):.2f}")

mid_btn.on_click(on_mid_search)

mid_tab = VBox([
    HBox([mid_inputs['middle_average'], mid_inputs['middle_strike_rate']]),
    mid_btn,
    mid_output
])

# Tab 4: Death Phase
death_inputs = {
    'death_average': FloatSlider(min=0, max=80, step=1, value=35, description='Death Average:'),
    'death_strike_rate': FloatSlider(min=0, max=200, step=5, value=145, description='Death Strike Rate:'),
}

death_output = Output()
death_btn = Button(description='Find Batters (distance < 3)', button_style='info')

def on_death_search(b):
    with death_output:
        death_output.clear_output()
        query = {k: v.value for k, v in death_inputs.items()}
        for col in feature_cols:
            if col not in query:
                query[col] = X[col].median()
        results = find_close_batters(query)
        
        if not results:
            print("No batters found within distance 2.5")
        else:
            print(f"Found {len(results)} batter(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Death Runs: {result.get('death_runs', 'N/A')}")
                print(f"   Death Average: {result.get('death_average', 'N/A'):.2f}")
                print(f"   Death Strike Rate: {result.get('death_strike_rate', 'N/A'):.2f}")

death_btn.on_click(on_death_search)

death_tab = VBox([
    HBox([death_inputs['death_average'], death_inputs['death_strike_rate']]),
    death_btn,
    death_output
])

# Create tabbed interface
tabs = Tab(
    children=[overall_tab, pp_tab, mid_tab, death_tab],
    titles=['Overall', 'Powerplay', 'Middle', 'Death']
)

display(tabs)

=== BATTER SEARCH BY PHASE ===


In [9]:
print("\n=== BOWLER SEARCH BY PHASE ===")

# Tab 1: Overall Stats (Runs, Econ, Avg, Wickets)
bowler_overall_inputs = {
    'total_runs': IntSlider(min=0, max=500, step=10, value=150, description='Total Runs:'),
    'economy': FloatSlider(min=0, max=15, step=0.25, value=8.5, description='Economy:'),
    'average': FloatSlider(min=0, max=100, step=1, value=30, description='Average:'),
    'wickets': IntSlider(min=0, max=50, step=1, value=5, description='Wickets:'),
}

bowler_overall_output = Output()
bowler_overall_btn = Button(description='Find Bowlers (distance < 2.5)', button_style='success')

def on_bowler_overall_search(b):
    with bowler_overall_output:
        bowler_overall_output.clear_output()
        query = {k: v.value for k, v in bowler_overall_inputs.items()}
        # Fill in other features with median
        for col in bowler_feature_cols:
            if col not in query:
                query[col] = X_bowlers[col].median()
        results = find_close_bowlers(query)
        
        if not results:
            print("No bowlers found within distance 2.5")
        else:
            print(f"Found {len(results)} bowler(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Total Runs: {result.get('total_runs', 'N/A')}")
                print(f"   Economy: {result.get('economy', 'N/A'):.2f}")
                print(f"   Average: {result.get('average', 'N/A'):.2f}")
                print(f"   Wickets: {result.get('wickets', 'N/A')}")

bowler_overall_btn.on_click(on_bowler_overall_search)

bowler_overall_tab = VBox([
    HBox([bowler_overall_inputs['total_runs'], bowler_overall_inputs['economy']]),
    HBox([bowler_overall_inputs['average'], bowler_overall_inputs['wickets']]),
    bowler_overall_btn,
    bowler_overall_output
])

# Tab 2: Powerplay Phase
bowler_pp_inputs = {
    'powerplay_economy': FloatSlider(min=0, max=15, step=0.25, value=9, description='PP Economy:'),
    'powerplay_average': FloatSlider(min=0, max=100, step=1, value=28, description='PP Average:'),
}

bowler_pp_output = Output()
bowler_pp_btn = Button(description='Find Bowlers (distance < 3)', button_style='success')

def on_bowler_pp_search(b):
    with bowler_pp_output:
        bowler_pp_output.clear_output()
        query = {k: v.value for k, v in bowler_pp_inputs.items()}
        for col in bowler_feature_cols:
            if col not in query:
                query[col] = X_bowlers[col].median()
        results = find_close_bowlers(query)
        
        if not results:
            print("No bowlers found within distance 2.5")
        else:
            print(f"Found {len(results)} bowler(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   PP Runs: {result.get('powerplay_runs', 'N/A')}")
                print(f"   PP Deliveries: {result.get('powerplay_deliveries', 'N/A')}")
                print(f"   PP Economy: {result.get('powerplay_economy', 'N/A'):.2f}")
                print(f"   PP Average: {result.get('powerplay_average', 'N/A'):.2f}")

bowler_pp_btn.on_click(on_bowler_pp_search)

bowler_pp_tab = VBox([
    HBox([bowler_pp_inputs['powerplay_economy'], bowler_pp_inputs['powerplay_average']]),
    bowler_pp_btn,
    bowler_pp_output
])

# Tab 3: Middle Phase
bowler_mid_inputs = {
    'middle_economy': FloatSlider(min=0, max=15, step=0.25, value=8, description='Middle Economy:'),
    'middle_average': FloatSlider(min=0, max=100, step=1, value=32, description='Middle Average:'),
}

bowler_mid_output = Output()
bowler_mid_btn = Button(description='Find Bowlers (distance < 3)', button_style='success')

def on_bowler_mid_search(b):
    with bowler_mid_output:
        bowler_mid_output.clear_output()
        query = {k: v.value for k, v in bowler_mid_inputs.items()}
        for col in bowler_feature_cols:
            if col not in query:
                query[col] = X_bowlers[col].median()
        results = find_close_bowlers(query)
        
        if not results:
            print("No bowlers found within distance 2.5")
        else:
            print(f"Found {len(results)} bowler(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Middle Runs: {result.get('middle_runs', 'N/A')}")
                print(f"   Middle Deliveries: {result.get('middle_deliveries', 'N/A')}")
                print(f"   Middle Economy: {result.get('middle_economy', 'N/A'):.2f}")
                print(f"   Middle Average: {result.get('middle_average', 'N/A'):.2f}")

bowler_mid_btn.on_click(on_bowler_mid_search)

bowler_mid_tab = VBox([
    HBox([bowler_mid_inputs['middle_economy'], bowler_mid_inputs['middle_average']]),
    bowler_mid_btn,
    bowler_mid_output
])

# Tab 4: Death Phase
bowler_death_inputs = {
    'death_economy': FloatSlider(min=0, max=15, step=0.25, value=9.5, description='Death Economy:'),
    'death_average': FloatSlider(min=0, max=100, step=1, value=25, description='Death Average:'),
}

bowler_death_output = Output()
bowler_death_btn = Button(description='Find Bowlers (distance < 3)', button_style='success')

def on_bowler_death_search(b):
    with bowler_death_output:
        bowler_death_output.clear_output()
        query = {k: v.value for k, v in bowler_death_inputs.items()}
        for col in bowler_feature_cols:
            if col not in query:
                query[col] = X_bowlers[col].median()
        results = find_close_bowlers(query)
        
        if not results:
            print("No bowlers found within distance 2.5")
        else:
            print(f"Found {len(results)} bowler(s)")
            print("\n" + "="*80)
            for i, result in enumerate(results, 1):
                print(f"\n{i}. {result['player']}")
                print(f"   Distance: {result['distance']:.4f}")
                print(f"   Cluster: {result.get('cluster', 'N/A')}")
                print(f"   Death Runs: {result.get('death_runs', 'N/A')}")
                print(f"   Death Deliveries: {result.get('death_deliveries', 'N/A')}")
                print(f"   Death Economy: {result.get('death_economy', 'N/A'):.2f}")
                print(f"   Death Average: {result.get('death_average', 'N/A'):.2f}")

bowler_death_btn.on_click(on_bowler_death_search)

bowler_death_tab = VBox([
    HBox([bowler_death_inputs['death_economy'], bowler_death_inputs['death_average']]),
    bowler_death_btn,
    bowler_death_output
])

# Create tabbed interface for bowlers
bowler_tabs = Tab(
    children=[bowler_overall_tab, bowler_pp_tab, bowler_mid_tab, bowler_death_tab],
    titles=['Overall', 'Powerplay', 'Middle', 'Death']
)

display(bowler_tabs)


=== BOWLER SEARCH BY PHASE ===
